In [ ]:
# ==============================================================================
# Direct Multimodal LLM Preference Analysis & Evaluation Pipeline
# Per-Annotator Version: reads from ./annotator/[ID].csv, images from ./screen/
# Outputs to ./results/[ID].json
#
# ACCURACY IMPROVEMENTS APPLIED:
#   1. Phase A: Richer prompts — rejection reasoning, intensity signal, confidence
#   2. Phase C: Sort by strength before sampling, drop contradictory/low-evidence
#              criteria, exclude low-confidence features
#   3. Phase D: Relative/comparative scoring, confidence gating, margin-based
#              tiebreaking, majority-vote (3× calls), penalise non-discriminating
#              criteria
#   4. Sampling: Strength-stratified train (20 rows), test = all remaining rows
# ==============================================================================
import os
import json
import base64
import time
import random
import re
import urllib.request
import urllib.error
import http.client
import concurrent.futures
from typing import List, Dict, Any, Optional
from pathlib import Path
import pandas as pd
from PIL import Image
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ==============================================================================
# CONFIGURATION
# ==============================================================================
ANNOTATOR_DIR = "./annotator"
IMAGES_DIR = "./screen"
RESULTS_DIR = "./results_parallel_20_new"
MODEL = "gpt-5.4"
TRAIN_SIZE = 20        # Fixed rows for preference learning (Phase A + C)
                       # Test = everything else in the CSV
SEED = 7

# Thread Pool configuration for parallel LLM execution
MAX_WORKERS = 7        # Adjust based on your Azure OpenAI RPM/TPM limits

# Majority-vote repetitions in Phase D (odd number recommended)
PHASE_D_VOTE_RUNS = 3

# Score margin below which a criterion is considered non-discriminating
NON_DISCRIMINATING_MARGIN = 1.0

# Minimum absolute score gap between A and B totals to avoid coin-flip ties
TIE_MARGIN = 2.0

AZURE_OPENAI_TARGET_URI = os.getenv(
    "AZURE_OPENAI_TARGET_URI",
    f"https://wu-lab-east-us-2.openai.azure.com/openai/deployments/{MODEL}/chat/completions?api-version=2025-01-01-preview"
)
AZURE_OPENAI_API_KEY="XXX"

# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================
def set_seed(seed: int):
    random.seed(seed)

MAX_SIZE = (1024, 1024)

def encode_image(image_path: str) -> Optional[str]:
    try:
        img = Image.open(image_path).convert("RGB")
        img.thumbnail(MAX_SIZE, Image.Resampling.LANCZOS)
        import io
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=85)
        return base64.b64encode(buf.getvalue()).decode("utf-8").replace("\n", "")
    except Exception as e:
        print(f"  [!] Error encoding image {image_path}: {e}")
        return None

def extract_json_from_text(text: str) -> dict:
    try:
        match = re.search(r'```(?:json)?\s*(.*?)\s*```', text, re.DOTALL)
        if match:
            return json.loads(match.group(1))
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"  [!] JSON Parse Error. Raw text:\n{text}\n")
        raise e

def is_valid_phase_a_result(result: Any) -> bool:
    if not isinstance(result, dict):
        return False
    return isinstance(result.get("preferred_features"), list)

def is_valid_phase_d_result(result: Any) -> bool:
    if not isinstance(result, dict):
        return False
    return isinstance(result.get("criteria_evaluations"), list)

def call_llm(
    prompt: str,
    image_a_b64: Optional[str] = None,
    image_b_b64: Optional[str] = None,
    max_retries: int = 5,
    temperature: float = 0.2,
) -> Dict[str, Any]:
    content = [{"type": "text", "text": prompt}]
    if image_a_b64 and image_b_b64:
        content.extend([
            {"type": "text", "text": "Image A:"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_a_b64}"}},
            {"type": "text", "text": "Image B:"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b_b64}"}},
        ])
    payload = {
        "messages": [{"role": "user", "content": content}],
        "temperature": temperature,
        "max_completion_tokens": 1200,
    }
    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPENAI_API_KEY,
        "User-Agent": "Jupyter-LLM-Evaluator/1.0",
    }
    data = json.dumps(payload).encode("utf-8")
    for attempt in range(max_retries):
        try:
            req = urllib.request.Request(
                AZURE_OPENAI_TARGET_URI, data=data, headers=headers, method="POST"
            )
            with urllib.request.urlopen(req, timeout=30) as response:
                result = json.loads(response.read().decode("utf-8"))
                content_str = result["choices"][0]["message"]["content"]
                return extract_json_from_text(content_str)
        except urllib.error.HTTPError as e:
            try:
                body = e.read().decode("utf-8")
                print(f"  [!] HTTP {e.code} (Attempt {attempt+1}/{max_retries}): {body}")
            except Exception:
                print(f"  [!] HTTP {e.code} (Attempt {attempt+1}/{max_retries}): {e.reason}")
            if e.code == 400:
                raise
            if attempt < max_retries - 1:
                time.sleep(2 + (2 ** attempt))
            else:
                raise
        except (urllib.error.URLError, http.client.RemoteDisconnected, ConnectionResetError) as e:
            print(f"  [!] Connection Error (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 + (2 ** attempt))
            else:
                raise
        except Exception as e:
            print(f"  [!] API Error (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 + (2 ** attempt))
            else:
                raise
    return {}

# ==============================================================================
# DATA HELPERS
# ==============================================================================
def get_winner_label(row: pd.Series) -> str:
    choice = str(row.get("final_choice", "")).strip()
    if choice in ("A < B", "A << B"):
        return "B"
    return "A"

def get_preference_strength(row: pd.Series) -> int:
    choice = str(row.get("final_choice", "")).strip()
    if ">>" in choice or "<<" in choice:
        return 2
    if ">" in choice or "<" in choice:
        return 1
    return 1

def load_annotator_data(csv_path: str) -> pd.DataFrame:
    return pd.read_csv(csv_path)

def sample_and_split(df: pd.DataFrame, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Stratified sample of exactly TRAIN_SIZE rows for training.
    Test set = all remaining rows (df minus the train rows).
    """
    strong = df[df["final_choice"].str.contains(r">>|<<", na=False)]
    weak   = df[~df["final_choice"].str.contains(r">>|<<", na=False)]

    n_strong = min(len(strong), round(TRAIN_SIZE * len(strong) / len(df)))
    n_weak   = TRAIN_SIZE - n_strong

    sampled_strong = strong.sample(n=n_strong, random_state=seed) if n_strong > 0 else strong.iloc[:0]
    sampled_weak   = weak.sample(n=n_weak,     random_state=seed) if n_weak   > 0 else weak.iloc[:0]

    train = (
        pd.concat([sampled_strong, sampled_weak])
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )
    # Test is everything not selected for train, preserving original df index for drop
    train_original_indices = pd.concat([sampled_strong, sampled_weak]).index
    test = df.drop(index=train_original_indices).reset_index(drop=True)
    return train, test

# ==============================================================================
# PHASE A — PAIR ANALYSIS (FEATURE EXTRACTION)
# ==============================================================================
def phase_a(train_df: pd.DataFrame, annotator_id: str) -> list:
    print(f"\n  --- Phase A: Pair Analysis ({len(train_df)} pairs / {MAX_WORKERS} workers) ---")
    analysis_results = []

    def _worker(idx, row):
        cid = row.get("cid", "unknown")
        img_a_path = os.path.join(IMAGES_DIR, row["left_file"])
        img_b_path = os.path.join(IMAGES_DIR, row["right_file"])
        img_a_b64 = encode_image(img_a_path)
        img_b_b64 = encode_image(img_b_path)
        if not img_a_b64 or not img_b_b64:
            return {"skip": True, "reason": "missing image"}

        winner   = get_winner_label(row)
        loser    = "B" if winner == "A" else "A"
        preference_strength = get_preference_strength(row)
        final_choice_raw    = str(row.get("final_choice", "")).strip()

        intensity_note = (
            "This was a STRONG preference (the user was very decisive)."
            if preference_strength == 2
            else "This was a MILD preference (the user had a slight lean)."
        )

        ANALYSIS_PROMPT = f"""You are an expert UX design researcher discovering a user's NICHE taste.
The user looked at two UI designs: Design A and Design B.
The user's raw choice label was: "{final_choice_raw}"
The user EXPLICITLY CHOSE: Design {winner}. {intensity_note}

Look at both designs carefully.

1. Identify up to 7 design dimensions where they differ.
2. For each dimension, state the trait in the CHOSEN design ({winner}) vs the REJECTED design ({loser}).
3. Also write a "why_user_rejected_loser" summary — what specifically made the rejected design unacceptable.
4. For each feature, assign a confidence level: "high" if the difference is visually obvious and clearly aligns with the user's choice, "medium" if plausible, "low" if speculative.

Return ONLY a JSON object with this strict schema:
{{
  "why_user_chose_winner": "<max 15 words — specific visual reason>",
  "why_user_rejected_loser": "<max 15 words — what made Design {loser} unacceptable>",
  "preferred_features": [
    {{
      "dimension": "<snake_case_name>",
      "preferred_trait_in_winner": "<brief trait in Design {winner}>",
      "rejected_trait_in_loser": "<brief trait in Design {loser}>",
      "confidence": "<high|medium|low>"
    }}
  ]
}}"""

        try:
            result = call_llm(ANALYSIS_PROMPT, img_a_b64, img_b_b64)
            llm_output_valid = is_valid_phase_a_result(result)
            return {
                "skip": False,
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "human_choice": winner,
                "preference_strength": preference_strength,
                "final_choice_raw": final_choice_raw,
                "llm_output_valid": llm_output_valid,
                "llm_analysis": result,
            }
        except Exception as e:
            return {
                "skip": False,
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "human_choice": winner,
                "preference_strength": preference_strength,
                "final_choice_raw": final_choice_raw,
                "llm_output_valid": False,
                "llm_analysis": None,
                "error": str(e),
            }

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_worker, idx, row): (idx, row) for idx, row in train_df.iterrows()}
        for future in concurrent.futures.as_completed(futures):
            idx, row = futures[future]
            pair_num = idx + 1
            cid = row.get("cid", "unknown")
            try:
                res = future.result()
            except Exception as e:
                print(f"  -> Fatal Thread Error on {pair_num}: {e}")
                continue
            if res.get("skip"):
                print(f"  -> Skipping {pair_num} ({res.get('reason')})")
                continue
            clean_res = {k: v for k, v in res.items() if k != "skip"}
            analysis_results.append(clean_res)
            raw      = res.get("final_choice_raw")
            winner   = res.get("human_choice")
            strength = res.get("preference_strength")
            if "error" in res:
                print(f"  Analyzed {pair_num}/{len(train_df)} [cid={cid}] -> Failed: {res.get('error')}")
            else:
                print(
                    f"  Analyzed {pair_num}/{len(train_df)} [cid={cid}] "
                    f"[final_choice={raw}] [winner={winner}] [strength={strength}] -> Valid: {res.get('llm_output_valid')}"
                )
    return analysis_results

# ==============================================================================
# PHASE C — SYNTHESIS & DYNAMIC CRITERIA EXTRACTION (WEIGHTED RUBRIC)
# ==============================================================================
def phase_c(analysis_results: list) -> tuple[str, list]:
    print(f"\n  --- Phase C: Synthesis ({len(analysis_results)} analyses) ---")

    observations = []
    for a in analysis_results:
        strength = int(a.get("preference_strength", 1) or 1)
        raw_features = (
            a.get("llm_analysis", {}).get("preferred_features", [])
            if a.get("llm_analysis") else []
        )
        # Drop low-confidence features to reduce noise
        filtered_features = [
            f for f in raw_features
            if isinstance(f, dict) and f.get("confidence", "medium") != "low"
        ]
        weighted_features = [
            {**feature, "evidence_weight": strength}
            for feature in filtered_features
        ]
        observations.append({
            "final_choice_raw": a.get("final_choice_raw"),
            "preference_strength": strength,
            "preferred_features": weighted_features,
        })

    # Sort by strength descending so the top-50 sample is dominated by decisive pairs
    observations.sort(key=lambda x: x["preference_strength"], reverse=True)
    sample_observations = observations[:50]

    SYNTHESIS_PROMPT = f"""You are a UX researcher. You are given raw observations of what a specific user PREFERRED vs REJECTED across multiple UI pairs.

Raw Observations of Preferred vs Rejected traits (sorted strongest preference first):
{json.dumps(sample_observations, indent=2)}

Identify the strongest, most consistent patterns in what they PREFER.

Each observation includes `preference_strength` and each feature includes `evidence_weight`.
- weight 1 = weak preference (`<` or `>`)
- weight 2 = strong preference (`<<` or `>>`)

CRITICAL INSTRUCTIONS:
1. Identify exactly the TOP 7 evaluation criteria based strictly on WEIGHTED SUPPORT across observations.
   Group similar concepts, sum their weighted evidence, and assign a `weight` to each.
2. Strong-preference evidence (weight 2) must count MORE than weak-preference evidence (weight 1).
3. MINIMUM EVIDENCE: only include a criterion if it appears in AT LEAST 2 separate pairs.
   Criteria supported by only 1 pair are unreliable — discard them.
4. Identify any dimensions where the user appeared CONTRADICTORY (chose A for this trait sometimes and B other times).
   List them in "contradictory_signals" and DO NOT include them in the top 7 criteria.

Return a JSON object with this schema:
{{
  "niche_preference_profile": "<Clear, literal description of the specific visual traits this user likes. Be very concrete.>",
  "contradictory_signals": ["<dimension_name>", ...],
  "top_7_evaluation_criteria": [
    {{
      "criterion": "<snake_case_criterion>",
      "description": "<exact visual trait the user prefers for this dimension>",
      "weight": <numeric_float_based_on_weighted_support>
    }}
  ]
}}
"""

    print("  Synthesizing preference profile...")
    try:
        synthesis_result = call_llm(prompt=SYNTHESIS_PROMPT)
        niche_preference_profile = synthesis_result.get("niche_preference_profile", "Could not synthesize profile.")
        top_7_criteria = synthesis_result.get("top_7_evaluation_criteria", [])
        contradictory  = synthesis_result.get("contradictory_signals", [])
        if contradictory:
            print(f"  Contradictory dimensions excluded: {contradictory}")
        if not top_7_criteria:
            raise ValueError("No criteria extracted")
    except Exception as e:
        print(f"  -> Synthesis failed: {e}")
        niche_preference_profile = "Synthesis failed."
        top_7_criteria = []

    clean_top_7 = []
    for i, c in enumerate(top_7_criteria):
        if isinstance(c, dict):
            clean_top_7.append({
                "criterion": c.get("criterion", f"criterion_{i+1}"),
                "description": c.get("description", ""),
                "weight": float(c.get("weight", 1.0) if c.get("weight") is not None else 1.0),
            })
        else:
            clean_top_7.append({"criterion": f"criterion_{i+1}", "description": str(c), "weight": 1.0})

    if not clean_top_7:
        clean_top_7 = [{"criterion": f"fallback_{i+1}", "description": "fallback", "weight": 1.0} for i in range(7)]

    print(f"  Profile: {niche_preference_profile[:120]}...")
    for c in clean_top_7:
        print(f"    - {c['criterion']} (weight={c['weight']})")
    return niche_preference_profile, clean_top_7

# ==============================================================================
# PHASE D — EVALUATION (RELATIVE SCORING + MAJORITY VOTE)
# ==============================================================================
def _score_pair_once(
    img_a_b64: str,
    img_b_b64: str,
    niche_preference_profile: str,
    criteria_for_eval: list,
    weights_map: dict,
    temperature: float = 0.3,
) -> dict:
    """Single scoring pass. Returns dict with aggregate delta and raw evaluations."""
    criteria_json = json.dumps(criteria_for_eval, indent=2)

    EVAL_PROMPT = f"""You are an AI scoring two UI designs for a user with this EXACT preference profile:

USER'S NICHE DESIGN PREFERENCE PROFILE:
"{niche_preference_profile}"

Evaluation criteria (user's preferred trait per dimension):
{criteria_json}

CRITICAL INSTRUCTIONS:
1. For EACH criterion, assign a RELATIVE score: how much better does Design A satisfy this criterion
   compared to Design B, on a scale from -5 to +5.
   - Positive = A is better for this criterion
   - Negative = B is better for this criterion
   - 0 = effectively equal
2. Use the full range. Only assign 0 if the designs are genuinely indistinguishable on this criterion.
3. DO NOT use generic UI best practices. Score ONLY against the user's specific preferred trait.
4. Add a "confidence" field: "high" if the difference is visually clear, "medium" if uncertain, "low" if you are guessing.

Return ONLY a JSON object:
{{
  "criteria_evaluations": [
    {{
      "criterion": "<must match one of the 7 provided>",
      "relative_score_a_minus_b": <integer -5 to +5>,
      "reason": "<max 7 words>",
      "confidence": "<high|medium|low>"
    }}
  ]
}}
"""

    result = call_llm(EVAL_PROMPT, img_a_b64, img_b_b64, temperature=temperature)
    if not is_valid_phase_d_result(result):
        return {"valid": False, "total_delta": 0.0, "total_a": 0.0, "total_b": 0.0, "evaluations": []}

    total_delta = 0.0
    evaluations = []
    for eval_item in result.get("criteria_evaluations", []):
        crit_name  = eval_item.get("criterion", "")
        confidence = eval_item.get("confidence", "medium")
        try:
            delta = float(eval_item.get("relative_score_a_minus_b", 0))
        except (ValueError, TypeError):
            delta = 0.0

        # Zero out low-confidence and non-discriminating evaluations
        if confidence == "low" or abs(delta) < NON_DISCRIMINATING_MARGIN:
            effective_delta = 0.0
        else:
            effective_delta = delta

        weight = weights_map.get(crit_name, 1.0)
        weighted_delta = effective_delta * weight

        evaluations.append({
            **eval_item,
            "effective_delta": effective_delta,
            "applied_weight": weight,
            "weighted_delta": weighted_delta,
        })
        total_delta += weighted_delta

    return {
        "valid": True,
        "total_delta": total_delta,
        "total_a": max(total_delta, 0.0),
        "total_b": max(-total_delta, 0.0),
        "evaluations": evaluations,
    }


def phase_d(
    test_df: pd.DataFrame,
    niche_preference_profile: str,
    top_7_criteria: list,
) -> tuple[list, float, int, int]:
    print(f"\n  --- Phase D: Evaluation ({len(test_df)} pairs / {MAX_WORKERS} workers / {PHASE_D_VOTE_RUNS} votes each) ---")

    criteria_for_eval = [
        {"criterion": c["criterion"], "preferred_trait": c["description"]}
        for c in top_7_criteria
    ]
    weights_map = {c["criterion"]: float(c.get("weight", 1.0)) for c in top_7_criteria}

    def _worker(idx, row):
        cid = row.get("cid", "unknown")
        img_a_path = os.path.join(IMAGES_DIR, row["left_file"])
        img_b_path = os.path.join(IMAGES_DIR, row["right_file"])
        img_a_b64 = encode_image(img_a_path)
        img_b_b64 = encode_image(img_b_path)
        if not img_a_b64 or not img_b_b64:
            return {"skip": True, "reason": "missing image"}

        true_winner      = get_winner_label(row)
        final_choice_raw = str(row.get("final_choice", "")).strip()
        true_strength    = get_preference_strength(row)

        try:
            run_results = []
            for _ in range(PHASE_D_VOTE_RUNS):
                run = _score_pair_once(
                    img_a_b64, img_b_b64,
                    niche_preference_profile, criteria_for_eval, weights_map,
                    temperature=0.3,
                )
                run_results.append(run)

            valid_runs = [r for r in run_results if r["valid"]]
            llm_output_valid = len(valid_runs) > 0

            if not valid_runs:
                raise ValueError("All scoring runs produced invalid output")

            avg_delta     = sum(r["total_delta"] for r in valid_runs) / len(valid_runs)
            total_score_a = max(avg_delta, 0.0)
            total_score_b = max(-avg_delta, 0.0)

            votes_a   = sum(1 for r in valid_runs if r["total_delta"] > 0)
            votes_b   = sum(1 for r in valid_runs if r["total_delta"] < 0)
            votes_tie = len(valid_runs) - votes_a - votes_b

            # Use tie margin; fall back to vote count when too close to call
            if abs(avg_delta) < TIE_MARGIN:
                predicted_choice = "A" if votes_a >= votes_b else "B"
            else:
                predicted_choice = "A" if avg_delta > 0 else "B"

            is_correct = predicted_choice == true_winner

            best_run = max(valid_runs, key=lambda r: abs(r["total_delta"]))
            details = {
                "criteria_evaluations": best_run["evaluations"],
                "final_total_score_a": total_score_a,
                "final_total_score_b": total_score_b,
                "avg_delta": avg_delta,
                "votes_a": votes_a,
                "votes_b": votes_b,
                "votes_tie": votes_tie,
                "derived_predicted_choice": predicted_choice,
            }

            return {
                "skip": False,
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "true_winner": true_winner,
                "final_choice_raw": final_choice_raw,
                "true_strength": true_strength,
                "predicted_choice": predicted_choice,
                "is_correct": is_correct,
                "llm_output_valid": llm_output_valid,
                "details": details,
                "score_a": total_score_a,
                "score_b": total_score_b,
            }
        except Exception as e:
            return {
                "skip": False,
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "true_winner": true_winner,
                "final_choice_raw": final_choice_raw,
                "true_strength": true_strength,
                "predicted_choice": None,
                "is_correct": False,
                "llm_output_valid": False,
                "details": None,
                "error": str(e),
            }

    eval_predictions    = []
    correct_predictions = 0
    valid_evals         = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_worker, idx, row): (idx, row) for idx, row in test_df.iterrows()}
        for future in concurrent.futures.as_completed(futures):
            idx, row = futures[future]
            pair_num = idx + 1
            cid = row.get("cid", "unknown")
            try:
                res = future.result()
            except Exception as e:
                print(f"  -> Fatal Thread Error on {pair_num}: {e}")
                continue
            if res.get("skip"):
                print(f"  -> Skipping {pair_num} ({res.get('reason')})")
                continue

            clean_res = {k: v for k, v in res.items() if k not in ("skip", "score_a", "score_b")}
            eval_predictions.append(clean_res)

            true_winner = res.get("true_winner")
            predicted   = res.get("predicted_choice")
            is_correct  = res.get("is_correct")
            valid       = res.get("llm_output_valid")

            if valid:
                valid_evals += 1
                if is_correct:
                    correct_predictions += 1

            if "error" in res:
                print(f"  Eval {pair_num}/{len(test_df)} [cid={cid}] -> Failed: {res.get('error')}")
            else:
                running_acc = (correct_predictions / valid_evals) if valid_evals > 0 else 0.0
                verdict     = "RIGHT" if is_correct else "WRONG"
                score_a     = res.get("score_a", 0.0)
                score_b     = res.get("score_b", 0.0)
                details     = res.get("details", {}) or {}
                votes_a     = details.get("votes_a", "-")
                votes_b     = details.get("votes_b", "-")
                print(
                    f"  Eval {pair_num}/{len(test_df)} [cid={cid}] -> "
                    f"ΔA={score_a:.2f} ΔB={score_b:.2f} | votes={votes_a}A/{votes_b}B | "
                    f"Pred={predicted} | True={true_winner} | {verdict} | "
                    f"Parsed={valid} | Acc={correct_predictions}/{valid_evals} ({running_acc*100:.2f}%)"
                )

    final_accuracy = (correct_predictions / valid_evals) if valid_evals > 0 else 0.0
    print(f"  Final Accuracy: {correct_predictions}/{valid_evals} ({final_accuracy*100:.2f}%)")
    return eval_predictions, final_accuracy, correct_predictions, valid_evals

# ==============================================================================
# MAIN — ITERATE OVER ALL ANNOTATORS
# ==============================================================================
def run_pipeline_for_annotator(csv_path: str, annotator_id: str):
    print(f"\n{'='*60}")
    print(f"Processing annotator: {annotator_id}")
    print(f"{'='*60}")
    set_seed(SEED)

    df = load_annotator_data(csv_path)
    print(f"  Loaded {len(df)} rows from {csv_path}")

    if len(df) <= TRAIN_SIZE:
        print(f"  Warning: only {len(df)} rows — need more than {TRAIN_SIZE} to have a non-empty test set.")

    train_df, test_df = sample_and_split(df, seed=SEED)
    print(f"  Train: {len(train_df)} | Test: {len(test_df)} (all remaining rows)")

    # Phase A
    analysis_results = phase_a(train_df, annotator_id)
    # Phase C
    niche_preference_profile, top_7_criteria = phase_c(analysis_results)
    # Phase D
    eval_predictions, accuracy, correct, total_eval = phase_d(
        test_df, niche_preference_profile, top_7_criteria
    )

    output = {
        "annotator_id": annotator_id,
        "model": MODEL,
        "sample_sizes": {
            "total_rows": len(df),
            "train": len(train_df),
            "test": len(test_df),
        },
        "niche_preference_profile": niche_preference_profile,
        "top_7_criteria": top_7_criteria,
        "pair_analyses": analysis_results,
        "evaluation_metrics": {
            "accuracy": accuracy,
            "correct": correct,
            "total_evaluated": total_eval,
        },
        "predictions": eval_predictions,
    }

    os.makedirs(RESULTS_DIR, exist_ok=True)
    out_path = os.path.join(RESULTS_DIR, f"{annotator_id}.json")
    with open(out_path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\n  Saved → {out_path}")
    return output


def main():
    annotator_dir = Path(ANNOTATOR_DIR)
    if not annotator_dir.exists():
        print(f"Error: {ANNOTATOR_DIR} directory not found.")
        return
    csv_files = sorted(annotator_dir.glob("*.csv"))
    if not csv_files:
        print(f"No CSV files found in {ANNOTATOR_DIR}")
        return
    print(f"Found {len(csv_files)} annotator(s): {[f.stem for f in csv_files]}")
    for csv_path in csv_files:
        annotator_id = csv_path.stem
        try:
            run_pipeline_for_annotator(str(csv_path), annotator_id)
        except Exception as e:
            print(f"\n[ERROR] Failed for annotator {annotator_id}: {e}")
    print("\nAll annotators processed.")


if __name__ == "__main__":
    main()

Found 20 annotator(s): ['0c65ba0b46894372', '1d3ee9b46ac34e6c', '247b7dfa8a5347ad', '2c79548f2ab44243', '2d8219ac74d146ba', '454c297fa1e54296', '498b9ea72d994e8e', '653f05d5c9ab4c97', '6ccbe484cb96425f', '7602a5d37b7d4220', '7a945bb87f2b4cd2', '8287fcc5b6504e39', '8f61f5a0685f42ec', '97945b44a2b54914', 'a692cdb11bbe432c', 'abda3f52c24c4038', 'ad7def63e86045b1', 'dad4b876ad3147e4', 'e8f37a526c444958', 'eee1fd24aad648ed']

Processing annotator: 0c65ba0b46894372
  Loaded 610 rows from annotator/0c65ba0b46894372.csv
  Train: 20 | Test: 590 (all remaining rows)

  --- Phase A: Pair Analysis (20 pairs / 7 workers) ---
  Analyzed 4/20 [cid=50565::gpt_comp_s2w_50565_v1.png>>gpt_comp_s2w_50565_v2.png::T000354] [final_choice=A > B] [winner=A] [strength=1] -> Valid: True
  Analyzed 2/20 [cid=51339::gpt_comp_s2w_51339_v1.png>>gpt_comp_s2w_51339_v3.png::DUP_T000361] [final_choice=A > B] [winner=A] [strength=1] -> Valid: True
  Analyzed 6/20 [cid=63085::gemini_comp_s2w_63085_v1.png>>gemini_comp_s2w_

KeyboardInterrupt: 